# Week 4 Homework: Deep Diagnostics + Memo

**ECBS5200 — Practical Deep Learning Engineering**

The lab gave you the diagnostic toolkit. Now you apply it at depth.
You'll extend every lab section — all six slice axes (not just two),
temperature scaling (not just ECE measurement), qualitative error
analysis, per-class confusion drill-down, and bootstrap confidence
intervals.

Then you write the memo. The memo is where the week's learning objective
lives: given a bag of diagnostic tools and two models, can you say
something rigorous about which one to trust and for what?

**Expected time:** ~6 hours including the memo. If the code runs smoothly
first try you'll finish closer to 4.5, but budget for debugging (shape
mismatches, a bootstrap re-run you forgot to seed) and for rewriting the
memo. Intellectual honesty: this is a 5-6 hour week, not a 4-hour week.

**Point breakdown (memo):** 20 / 20 / 25 / 20 / 15 across five sections.
The confusion-matrix section (25 pts) carries the most weight.

**How to use this notebook:**
- **GUIDED** cells run as-is.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a NameError-producing placeholder.
- The memo is at the end. Write as you go so the observations are fresh.

## Kaggle setup

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → GPU T4

In [ ]:
import subprocess, sys, os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if os.path.exists('/kaggle/working'):
    os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "datasets", "accelerate", "scikit-learn",
    "matplotlib", "pandas", "tqdm", "peft",
])
print("Packages installed.")

from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)

In [ ]:
import gc
import re
import time
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import (
    load_course_data, LABEL_LIST, NUM_LABELS, LABEL_TO_ID, ID_TO_LABEL, MAX_LENGTH,
)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    USE_BF16 = cc[0] >= 8
    USE_FP16 = not USE_BF16
    print(f"GPU: {torch.cuda.get_device_name(0)}  Precision: {'bf16' if USE_BF16 else 'fp16'}")

---
## Part 1: Load Data + Predictions (~10 min)

You saved `week4_lab_predictions.npz` from the lab. Upload it as a Kaggle
dataset attached to this notebook, or set `RERUN_INFERENCE = True` to
recompute the predictions from the HF Hub checkpoints (~1 min extra).
---

In [ ]:
# GUIDED: Load the canonical dataset
print("Loading dataset...")
train_ds, val_ds, _ = load_course_data()
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Classes: {NUM_LABELS}")

In [ ]:
# GUIDED: Try to load lab predictions; recompute if missing
RERUN_INFERENCE = False   # Set True if you don't have week4_lab_predictions.npz

lab_preds_path = Path("week4_lab_predictions.npz")
if lab_preds_path.exists() and not RERUN_INFERENCE:
    arr = np.load(lab_preds_path)
    enc = {"preds": arr["encoder_preds"], "labels": arr["encoder_labels"],
           "logits": arr["encoder_logits"], "max_conf": arr["encoder_max_conf"]}
    dec = {"preds": arr["decoder_preds"], "labels": arr["decoder_labels"],
           "logits": arr["decoder_logits"], "max_conf": arr["decoder_max_conf"]}
    enc["probs"] = torch.softmax(torch.from_numpy(enc["logits"]), dim=-1).numpy()
    dec["probs"] = torch.softmax(torch.from_numpy(dec["logits"]), dim=-1).numpy()
    print("Loaded predictions from lab.")
else:
    print("Re-running inference from HF Hub checkpoints...")
    ENCODER_REPO = "earino/ecbs5200-week3-encoder-lora"
    DECODER_REPO = "earino/ecbs5200-week3-decoder-lora"

    def run_inference(model, tokenizer, dataset):
        def tokenize(batch):
            return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)
        ds_tok = dataset.map(tokenize, batched=True, remove_columns=["text", "label_name"])
        ds_tok = ds_tok.rename_column("label", "labels")
        collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)
        trainer = Trainer(
            model=model,
            args=TrainingArguments(output_dir="/tmp/eval", per_device_eval_batch_size=64,
                                   report_to="none"),
            data_collator=collator,
        )
        out = trainer.predict(ds_tok)
        probs = torch.softmax(torch.from_numpy(out.predictions), dim=-1).numpy()
        return {
            "logits": out.predictions.astype(np.float32),
            "probs": probs.astype(np.float32),
            "preds": probs.argmax(-1).astype(np.int64),
            "labels": out.label_ids.astype(np.int64),
            "max_conf": probs.max(-1).astype(np.float32),
        }

    enc_tok = AutoTokenizer.from_pretrained(ENCODER_REPO)
    enc_mdl = AutoModelForSequenceClassification.from_pretrained(
        ENCODER_REPO, attn_implementation="sdpa").to(device).eval()
    enc = run_inference(enc_mdl, enc_tok, val_ds)
    del enc_mdl
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

    dec_tok = AutoTokenizer.from_pretrained(DECODER_REPO)
    dec_tok.padding_side = "left"
    if dec_tok.pad_token is None: dec_tok.pad_token = dec_tok.eos_token
    dec_mdl = AutoModelForSequenceClassification.from_pretrained(DECODER_REPO).to(device).eval()
    if dec_mdl.config.pad_token_id is None:
        dec_mdl.config.pad_token_id = dec_tok.pad_token_id
    dec = run_inference(dec_mdl, dec_tok, val_ds)
    del dec_mdl
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print("Predictions recomputed.")

print(f"Encoder: acc {(enc['preds'] == enc['labels']).mean():.4f}  "
      f"F1 {f1_score(enc['labels'], enc['preds'], average='macro', zero_division=0):.4f}")
print(f"Decoder: acc {(dec['preds'] == dec['labels']).mean():.4f}  "
      f"F1 {f1_score(dec['labels'], dec['preds'], average='macro', zero_division=0):.4f}")

---
## Part 2: All Six Slice Axes (~30 min)

The lab covered **char_length** and **redaction**. Now add four more.
Compute per-slice macro F1 for all six axes, both models. In your memo
you'll discuss TWO of them — the one with the strongest signal and
the null-result axis.
---

In [ ]:
# GUIDED: Axis 1 — char_length (from lab)
texts = val_ds["text"]
val_char_length = np.array([len(t) for t in texts])
char_q = np.percentile(val_char_length, [25, 50, 75])

def char_bucket(c):
    if c <= char_q[0]: return "q1_shortest"
    if c <= char_q[1]: return "q2"
    if c <= char_q[2]: return "q3"
    return "q4_longest"
axis_char = np.array([char_bucket(c) for c in val_char_length])

In [ ]:
# GUIDED: Axis 2 — redaction (from lab)
XXXX_RE = re.compile(r"X{4}")
axis_redact = np.array(["redacted" if XXXX_RE.search(t) else "clean" for t in texts])

In [ ]:
# INTERACTIVE: Axis 3 — token length (encoder tokenizer, bucketed)
# Tokenize each val example WITHOUT truncation to get full token length.
# Bucket: 1-30 / 31-80 / 81-127 / truncated-at-128 (length > 128).

enc_tokenizer = AutoTokenizer.from_pretrained("earino/ecbs5200-week3-encoder-lora")

# Batched tokenization — much faster than per-example
tokenized = enc_tokenizer(list(texts), truncation=False, padding=False)
val_token_lengths = np.array([len(ids) for ids in tokenized["input_ids"]])

def tok_bucket(L):
    if L <= 30: return "1-30"
    if L <= 80: return "31-80"
    if L < 128: return "81-127"
    return "truncated_128"

axis_tok = np.array([tok_bucket(L) for L in val_token_lengths])

In [ ]:
# INTERACTIVE: Axis 4 — numeric content
# Tag as "numeric" if the text contains a dollar sign followed by a digit
# OR a standalone sequence of 4+ digits. Otherwise "non_numeric".
# Regex to use: r"\$\d|(?<!\w)\d{4,}(?!\w)"

NUM_RE = re.compile(r"\$\d|(?<!\w)\d{4,}(?!\w)")
axis_numeric = np.array(["numeric" if ___ else "non_numeric" for t in texts])

In [ ]:
# INTERACTIVE: Axis 5 — opener-I (first word is "I", personal framing)
# Tag as "starts_I" if text (after leading whitespace) starts with "I "
# or "I'". Otherwise "other_opener".

def starts_with_I(t):
    s = t.lstrip()
    return ___  # your condition

axis_opener = np.array(["starts_I" if starts_with_I(t) else "other_opener" for t in texts])

In [ ]:
# INTERACTIVE: Axis 6 — all-caps word fraction
# Count words that are fully uppercase (3+ letters). Divide by total
# word count for each text. Median-split into "caps_low" vs "caps_high".

ALLCAPS_WORD = re.compile(r"\b[A-Z]{3,}\b")

def caps_frac(t):
    words = t.split()
    if not words: return 0.0
    return ___ / max(1, len(words))   # count of all-caps words / total words

caps_fractions = np.array([caps_frac(t) for t in texts])
caps_median = float(np.median(caps_fractions))
axis_caps = np.array(["caps_low" if c <= caps_median else "caps_high"
                      for c in caps_fractions])

In [ ]:
# GUIDED: Compute per-slice F1 for all axes, both models
all_axes = {
    "char_length":  axis_char,
    "redaction":    axis_redact,
    "tok_length":   axis_tok,
    "numeric":      axis_numeric,
    "opener_I":     axis_opener,
    "allcaps":      axis_caps,
}

rows = []
for axis_name, axis_vals in all_axes.items():
    for slice_name in sorted(set(axis_vals)):
        mask = axis_vals == slice_name
        if mask.sum() == 0: continue
        enc_f1 = f1_score(enc["labels"][mask], enc["preds"][mask],
                          average="macro", zero_division=0)
        dec_f1 = f1_score(dec["labels"][mask], dec["preds"][mask],
                          average="macro", zero_division=0)
        rows.append({
            "axis": axis_name, "slice": slice_name, "n": int(mask.sum()),
            "encoder_f1": round(enc_f1, 4), "decoder_f1": round(dec_f1, 4),
            "diff": round(dec_f1 - enc_f1, 4),
        })

slice_df = pd.DataFrame(rows)
print(slice_df.to_string(index=False))

In [ ]:
# GUIDED: Which axes produce the biggest encoder-decoder gaps?
axis_summary = slice_df.groupby("axis").apply(
    lambda g: pd.Series({
        "max_abs_diff": g["diff"].abs().max(),
        "slices":       len(g),
    }), include_groups=False,
).sort_values("max_abs_diff", ascending=False)
print("\nAxes ranked by maximum |encoder-decoder| gap:")
print(axis_summary)

### INTERACTIVE: Interpretation — strongest-signal axis

Pick the axis with the biggest encoder-decoder gaps. Which slice of
that axis shows the biggest gap, and what's a plausible story for WHY
the decoder wins bigger (or smaller) on that slice? Cite the specific
F1 numbers.

YOUR ANSWER (~4-6 sentences, with numbers):




### INTERACTIVE: Interpretation — the null-result axis (opener_I)

Look at the opener_I rows. The encoder-decoder gap on both "starts_I"
and "other_opener" should be almost identical — this axis shows
essentially no differential signal.

**A null result is still a result.** What does it MEAN when an axis
shows no differential signal between the two models? Why should you
still report a null-result axis in diagnostic work instead of
quietly dropping it?

YOUR ANSWER (~3-4 sentences):




---
## Part 3: Temperature Scaling Experiment (~30 min)

The lab measured ECE on both models. You saw the decoder was MORE
overconfident than the encoder (unexpected). Now apply the standard
fix: temperature scaling.

**Protocol:**
1. Split val 50/50 into a calibration fold and an eval fold.
2. On the calibration fold, find the scalar T that minimizes NLL when
   logits are divided by T: `softmax(logits / T)`.
3. Apply T to the eval fold. Measure ECE before and after scaling.
4. Also measure macro F1 before and after — temperature scaling should
   NOT change argmax (and thus not change macro F1).
---

In [ ]:
# GUIDED: Helpers (compute_ece and fit_temperature)
def compute_ece(confidences, correct, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(confidences, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0: continue
        ece += (mask.sum() / len(confidences)) * abs(
            confidences[mask].mean() - correct[mask].mean())
    return float(ece)

def fit_temperature(logits, labels, max_iter=200):
    """LBFGS optimization of scalar T to minimize NLL."""
    logits_t = torch.from_numpy(logits).float()
    labels_t = torch.from_numpy(labels).long()
    T = torch.nn.Parameter(torch.ones(1) * 1.5)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=max_iter)
    nll = torch.nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        loss = nll(logits_t / T, labels_t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(T.detach().cpu().item())

In [ ]:
# INTERACTIVE: Build the 50/50 fold split
# Deterministic — use SEED + a fixed offset so everyone gets the same split.

n_val = len(enc["labels"])
rng_fold = np.random.RandomState(SEED + 999)
perm = rng_fold.permutation(n_val)
cal_idx = perm[: n_val // 2]
eval_idx = perm[n_val // 2 :]

print(f"Calibration fold: {len(cal_idx)} examples")
print(f"Eval fold:        {len(eval_idx)} examples")

In [ ]:
# INTERACTIVE: Fit T on cal fold, evaluate on eval fold, both models

results_temp = {}
for model_name, pack in [("encoder", enc), ("decoder", dec)]:
    logits = pack["logits"]
    labels = pack["labels"]

    # Pre-scaling ECE on eval fold
    correct_eval = (pack["preds"][eval_idx] == labels[eval_idx]).astype(np.float32)
    ece_pre = compute_ece(pack["max_conf"][eval_idx], correct_eval, n_bins=15)
    f1_pre  = f1_score(labels[eval_idx], pack["preds"][eval_idx],
                       average="macro", zero_division=0)

    # Fit T on calibration fold
    T = fit_temperature(logits[cal_idx], labels[cal_idx])

    # Apply T to eval fold, recompute
    scaled_probs = torch.softmax(torch.from_numpy(logits[eval_idx]) / T, dim=-1).numpy()
    scaled_preds = scaled_probs.argmax(-1)
    scaled_conf  = scaled_probs.max(-1)
    correct_post = (scaled_preds == labels[eval_idx]).astype(np.float32)
    ece_post = compute_ece(scaled_conf, correct_post, n_bins=15)
    f1_post  = f1_score(labels[eval_idx], scaled_preds,
                        average="macro", zero_division=0)

    results_temp[model_name] = {
        "T": T, "ece_pre": ece_pre, "ece_post": ece_post,
        "f1_pre": float(f1_pre), "f1_post": float(f1_post),
    }

    print(f"{model_name.upper()}")
    print(f"  T = {T:.3f}")
    print(f"  ECE:      {ece_pre:.4f}  →  {ece_post:.4f}  "
          f"(reduction: {(1-ece_post/ece_pre)*100:.1f}%)")
    print(f"  Macro F1: {f1_pre:.4f}  →  {f1_post:.4f}  "
          f"(should be identical — argmax is T-invariant)")
    print()

### INTERACTIVE: Does temperature scaling change the ranking?

Before scaling, the decoder had higher ECE than the encoder (worse
calibrated). After scaling, is the decoder still more miscalibrated,
or does temperature scaling flip the ranking?

What does your answer suggest about how to RANK models on calibration
— pre-scaling or post-scaling?

YOUR ANSWER (~3-4 sentences):




---
## Part 4: Qualitative Error Analysis (~30 min)

Pull examples where the two models DISAGREE — specifically where one
is right and the other is wrong. Tag each example with a 2-3 word
descriptor, then look for patterns.

**Why this matters:** aggregate numbers tell you THAT the models differ.
Qualitative analysis tells you WHAT KIND of examples drive the
difference. That's the input to Week 6's distillation work — a
disagreement dataset with semantic structure.
---

In [ ]:
# GUIDED: Find disagreement examples
encoder_right_decoder_wrong = np.where(
    (enc["preds"] == enc["labels"]) & (dec["preds"] != dec["labels"])
)[0]
decoder_right_encoder_wrong = np.where(
    (dec["preds"] == dec["labels"]) & (enc["preds"] != enc["labels"])
)[0]
print(f"Encoder right / decoder wrong: {len(encoder_right_decoder_wrong)}")
print(f"Decoder right / encoder wrong: {len(decoder_right_encoder_wrong)}")

In [ ]:
# GUIDED: Sample 20 from each direction (deterministic)
rng_sample = np.random.RandomState(SEED + 111)
enc_right_sample = rng_sample.choice(encoder_right_decoder_wrong, size=20, replace=False)
dec_right_sample = rng_sample.choice(decoder_right_encoder_wrong, size=20, replace=False)

sample_idx = np.concatenate([enc_right_sample, dec_right_sample])
sample_df = pd.DataFrame({
    "example_idx":   sample_idx,
    "who_is_right":  ["encoder"] * 20 + ["decoder"] * 20,
    "text":          [val_ds["text"][i][:200] for i in sample_idx],
    "true_class":    [LABEL_LIST[int(enc["labels"][i])] for i in sample_idx],
    "encoder_pred":  [LABEL_LIST[int(enc["preds"][i])]  for i in sample_idx],
    "decoder_pred":  [LABEL_LIST[int(dec["preds"][i])]  for i in sample_idx],
    "your_tag":      [""] * 40,
})
print("Sample built — 40 rows ready for tagging.")

### INTERACTIVE: Tag each example

For each of the 40 rows, read the text + predictions and write a 2-3
word descriptor in the `your_tag` column. Some starter categories
(you can invent your own):

- `"short complaint"` — very brief, 1-2 sentences
- `"numeric-heavy"` — lots of dollar amounts or dates
- `"legal jargon"` — mentions specific laws, acts, or procedures
- `"emotional tone"` — ALL CAPS, exclamation marks, anger
- `"ambiguous product"` — the product category itself is unclear
- `"specific vendor"` — names a specific bank or service
- `"off-topic"` — the complaint doesn't seem to match the true label

Fill the `your_tag` column. You can see/edit the DataFrame below.

In [ ]:
# GUIDED: Print all 40 examples. Read them, then tag each in the next cell.
for i, row in sample_df.iterrows():
    print(f"[{i:>2}] {row['who_is_right']:>8} right | true: {row['true_class'][:30]}")
    print(f"     enc pred: {row['encoder_pred'][:40]}")
    print(f"     dec pred: {row['decoder_pred'][:40]}")
    print(f"     text: {row['text'][:180]}")
    print()

In [ ]:
# INTERACTIVE: Actually write your tags here
# Replace these None values with your 2-3 word strings. All 40 rows.

tags = [
    None,  # 0 — encoder right, decoder wrong
    None,  # 1
    None,  # 2
    None,  # 3
    None,  # 4
    None,  # 5
    None,  # 6
    None,  # 7
    None,  # 8
    None,  # 9
    None,  # 10
    None,  # 11
    None,  # 12
    None,  # 13
    None,  # 14
    None,  # 15
    None,  # 16
    None,  # 17
    None,  # 18
    None,  # 19
    None,  # 20 — decoder right, encoder wrong
    None,  # 21
    None,  # 22
    None,  # 23
    None,  # 24
    None,  # 25
    None,  # 26
    None,  # 27
    None,  # 28
    None,  # 29
    None,  # 30
    None,  # 31
    None,  # 32
    None,  # 33
    None,  # 34
    None,  # 35
    None,  # 36
    None,  # 37
    None,  # 38
    None,  # 39
]

assert all(t is not None for t in tags), "All 40 tags must be filled before you proceed."
sample_df["your_tag"] = tags

In [ ]:
# GUIDED: Tally tag frequencies by direction
print("TAG FREQUENCIES — encoder right / decoder wrong (top 8):")
tag_counts_enc = Counter(sample_df[sample_df["who_is_right"] == "encoder"]["your_tag"])
for tag, count in tag_counts_enc.most_common(8):
    print(f"  {tag:<30} {count}")

print("\nTAG FREQUENCIES — decoder right / encoder wrong (top 8):")
tag_counts_dec = Counter(sample_df[sample_df["who_is_right"] == "decoder"]["your_tag"])
for tag, count in tag_counts_dec.most_common(8):
    print(f"  {tag:<30} {count}")

### INTERACTIVE: What patterns appear?

1. Which 1-2 tags dominate each direction? Are they the same or different?

2. Is there a coherent story for what KINDS of complaints the encoder
   handles better than the decoder (if any)? What about the decoder's
   wins?

3. Based on the tag patterns, if you had to write a "trigger rule"
   to route complaints — "send to encoder if X, send to decoder if Y"
   — what would X and Y be? (This is a preview of Week 5/6 routing
   and distillation logic.)

YOUR ANSWERS:

1.

2.

3.



---
## Part 5: Per-Class Confusion Drill-Down (~15 min)

For each model, find the 3 classes with the worst per-class F1
(skipping classes with 0 val examples). For each of those classes,
find the TOP 3 confusers — the classes the model predicts most often
when the true class is one of the worst.
---

In [ ]:
# GUIDED: Compute per-class F1 for both models
def per_class_f1(preds, labels):
    return f1_score(labels, preds, average=None, zero_division=0,
                    labels=list(range(NUM_LABELS)))

enc_pcf1 = per_class_f1(enc["preds"], enc["labels"])
dec_pcf1 = per_class_f1(dec["preds"], dec["labels"])

In [ ]:
# GUIDED: Val counts per class. We filter to classes with n>=5 val examples
# so the per-class F1 estimate is stable enough for pattern analysis. A
# class with n=1 val example and F1=0 is "worst" only because of resampling
# noise — that's a Part 6 bootstrap topic, not a confusion-pattern topic.
val_label_counts = Counter(val_ds["label"])
val_counts = np.array([val_label_counts.get(i, 0) for i in range(NUM_LABELS)])
present = val_counts >= 5

In [ ]:
# INTERACTIVE: Find the 3 worst-F1 classes per model (skip n=0 classes)
# Hint: argsort among classes with val_counts > 0, take the 3 lowest F1.

def worst_k_classes(per_class_f1_arr, k=3):
    candidates = [i for i in range(NUM_LABELS) if present[i]]
    # Sort candidates by F1 ascending, take k lowest
    sorted_candidates = sorted(candidates, key=lambda i: per_class_f1_arr[i])
    return sorted_candidates[:k]

enc_worst = worst_k_classes(enc_pcf1, k=3)
dec_worst = worst_k_classes(dec_pcf1, k=3)

print("ENCODER — 3 worst-F1 classes (with val examples):")
for c in enc_worst:
    print(f"  [{c:3d}] {LABEL_LIST[c][:60]}  "
          f"train={sum(1 for x in train_ds['label'] if x == c):5}  "
          f"val={val_counts[c]}  F1={enc_pcf1[c]:.3f}")

print("\nDECODER — 3 worst-F1 classes (with val examples):")
for c in dec_worst:
    print(f"  [{c:3d}] {LABEL_LIST[c][:60]}  "
          f"train={sum(1 for x in train_ds['label'] if x == c):5}  "
          f"val={val_counts[c]}  F1={dec_pcf1[c]:.3f}")

In [ ]:
# INTERACTIVE: For each worst class, find top-3 confusers
# Use the confusion matrix: cm[true_class, predicted_class] = count.
# For each worst true class, sort predicted-class counts (excluding the
# true class itself), take top 3.

cm_enc = confusion_matrix(enc["labels"], enc["preds"], labels=list(range(NUM_LABELS)))
cm_dec = confusion_matrix(dec["labels"], dec["preds"], labels=list(range(NUM_LABELS)))

def top_confusers(cm, true_class, k=3):
    row = cm[true_class].copy()
    row[true_class] = 0   # zero out the diagonal (correct predictions)
    top_k = np.argsort(row)[::-1][:k]
    return [(int(c), int(row[c])) for c in top_k if row[c] > 0]

print("=" * 70)
print("ENCODER — where do its worst-F1 classes get confused?")
print("=" * 70)
for c in enc_worst:
    print(f"\n[{c}] TRUE: {LABEL_LIST[c]}")
    for pred_c, count in top_confusers(cm_enc, c, k=3):
        print(f"      → [{pred_c:3d}] {LABEL_LIST[pred_c][:55]} ({count} times)")

print("\n" + "=" * 70)
print("DECODER — where do its worst-F1 classes get confused?")
print("=" * 70)
for c in dec_worst:
    print(f"\n[{c}] TRUE: {LABEL_LIST[c]}")
    for pred_c, count in top_confusers(cm_dec, c, k=3):
        print(f"      → [{pred_c:3d}] {LABEL_LIST[pred_c][:55]} ({count} times)")

### INTERACTIVE: Is the confusion pattern systematic?

Look at the true-class names and their top confusers. Are the
confusions mostly:

- **Semantically similar** (e.g., "Closing an account" confused with
  "Managing an account" — same product family)?
- **Frequency-similar** (rare confused with other rare, or rare
  confused with very common)?
- **Pipeline-induced** (artifacts of our merge mapping or filter —
  e.g., "Advertising" vs "Advertising and marketing" are likely the
  same CFPB concept we failed to merge; see the opening pipeline
  module from lecture)?
- **Arbitrary** (no obvious pattern)?

Write 3-5 sentences characterizing the confusion pattern you observe.
Cite 2-3 specific class pairs to support your characterization, and
**flag at least one pair that looks pipeline-induced** if you find one.

YOUR ANSWER:




---
## Part 6: Bootstrap Confidence Intervals (~30 min)

The encoder and decoder macro F1 numbers differ by ~0.03. Is that real?
Or would you see a 0.03 gap just by resampling a different val set?

Bootstrap answers this. Resample val indices WITH REPLACEMENT 1000
times. For each resample, compute macro F1 for each model AND their
difference. Report 95% confidence intervals for all three quantities.

**The key test:** does the CI for (decoder F1 − encoder F1) exclude zero?
If yes, the gap is real within your noise floor. If no, you can't
distinguish the decoder's advantage from resampling noise.
---

In [ ]:
# GUIDED: Bootstrap loop
N_BOOT = 1000
rng_boot = np.random.RandomState(SEED + 7)
n_val = len(enc["labels"])

boot_enc = np.zeros(N_BOOT)
boot_dec = np.zeros(N_BOOT)
boot_diff = np.zeros(N_BOOT)

print(f"Running {N_BOOT} bootstrap resamples...")
t0 = time.time()
for i in range(N_BOOT):
    idx = rng_boot.randint(0, n_val, size=n_val)
    f_e = f1_score(enc["labels"][idx], enc["preds"][idx],
                   average="macro", zero_division=0)
    f_d = f1_score(dec["labels"][idx], dec["preds"][idx],
                   average="macro", zero_division=0)
    boot_enc[i]  = f_e
    boot_dec[i]  = f_d
    boot_diff[i] = f_d - f_e
    if (i + 1) % 250 == 0:
        print(f"  {i+1}/{N_BOOT}  elapsed {time.time()-t0:.1f}s")

In [ ]:
# INTERACTIVE: Compute 95% CIs
# Hint: 2.5th and 97.5th percentiles of the bootstrap arrays.

def ci95(arr):
    lo = ___   # 2.5th percentile
    hi = ___   # 97.5th percentile
    return float(lo), float(hi)

enc_lo, enc_hi = ci95(boot_enc)
dec_lo, dec_hi = ci95(boot_dec)
diff_lo, diff_hi = ci95(boot_diff)
diff_excludes_zero = (diff_lo > 0) or (diff_hi < 0)

print(f"\nBootstrap results ({N_BOOT} resamples, 95% CI):")
print(f"  Encoder macro F1:   {boot_enc.mean():.4f}  [{enc_lo:.4f}, {enc_hi:.4f}]")
print(f"  Decoder macro F1:   {boot_dec.mean():.4f}  [{dec_lo:.4f}, {dec_hi:.4f}]")
print(f"  Difference:         {boot_diff.mean():+.4f}  [{diff_lo:+.4f}, {diff_hi:+.4f}]")
print(f"  Difference CI excludes zero: {diff_excludes_zero}")
print(f"  Fraction of resamples with decoder > encoder: "
      f"{(boot_diff > 0).mean()*100:.1f}%")

In [ ]:
# GUIDED: Overlaid histograms
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(boot_enc, bins=40, alpha=0.55, color="#3498db",
        label=f"encoder (mean={boot_enc.mean():.3f})")
ax.hist(boot_dec, bins=40, alpha=0.55, color="#e74c3c",
        label=f"decoder (mean={boot_dec.mean():.3f})")
ax.axvline(boot_enc.mean(), color="#3498db", linestyle="--", alpha=0.7)
ax.axvline(boot_dec.mean(), color="#e74c3c", linestyle="--", alpha=0.7)
ax.set_xlabel("macro F1 (bootstrap resamples)")
ax.set_ylabel("count")
ax.set_title(f"Bootstrap macro F1 — diff CI [{diff_lo:+.3f}, {diff_hi:+.3f}] "
             f"({'excludes' if diff_excludes_zero else 'includes'} 0)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### INTERACTIVE: Is the gap real?

1. Does the difference CI exclude zero? If so, you have evidence the
   decoder's advantage is real within resampling noise.

2. How wide is the CI? A CI of [+0.010, +0.048] says the decoder's
   advantage is "somewhere between tiny and substantial." What does
   a wide CI tell you about how confident you should be in a specific
   point estimate?

3. Recall that 17 val classes have n=1 and 35 have n≤4. How does that
   noise floor show up in the width of the bootstrap CIs?

YOUR ANSWERS:

1.

2.

3.



---
## Part 7: Memo (100 points)

Five sections, each aligned to a rubric criterion. Write each section
based on the evidence you generated above. Cite specific numbers.
When you don't know, say so — "I don't know" with noise-floor
justification scores better than an overclaim.
---

### MEMO SECTION 1: Slice analysis — where do the models differ? (20 points)

> Across the six slice axes you computed, where does the encoder-decoder
> gap open up, where does it close, and what do those differences tell
> you about which model handles which kind of input better? Include
> the null-result axis — a null result is evidence too.

**Evidence expected:** specific axis names, specific slice labels, F1
numbers on each.

**YOUR RESPONSE:**




### MEMO SECTION 2: Calibration — which model can you trust? (20 points)

> Which model is better-calibrated before temperature scaling? After?
> Does temperature scaling change the ranking? What does your answer
> imply for how you should interpret a model's confidence score in
> production?

**Evidence expected:** pre and post ECE values, T values for both
models, macro F1 invariance under scaling.

**YOUR RESPONSE:**




### MEMO SECTION 3: Confusion-matrix insight (25 points)

> When a rare-class complaint is misclassified, where does the wrong
> prediction actually land? What does the per-class confusion drill-down
> reveal about the *kinds* of confusions the models make? Given all
> this evidence, what does "fixing rare-class performance" actually
> mean operationally — and does it look tractable?

**Evidence expected:** tier-breakdown percentages from the lab (tail →
head / mid / tail-other), specific class pairs from Part 5, at least
one claim about whether the confusion pattern is "semantic
similarity," "frequency similarity," or something else.

**YOUR RESPONSE:**




### MEMO SECTION 4: Is the gap real within your noise floor? (20 points)

> The aggregate decoder-minus-encoder macro F1 is positive. Is that
> a real effect or could it plausibly be resampling noise? Defend your
> answer with the bootstrap CI. Also: how does the n=1 val-class
> cluster (17 classes with just one val example) affect how much
> weight you put on a specific macro F1 point estimate?

**Evidence expected:** bootstrap CI for the difference, fraction of
resamples where decoder > encoder, discussion of val-reliability
noise.

**YOUR RESPONSE:**




### MEMO SECTION 5: Your confidence in the Week 3 deployment recommendation (15 points)

> At the end of Week 3, you (or your classmates) made a deployment
> recommendation — encoder or decoder, and why. Given everything you
> now know from a week of diagnostics:
>
> - Has your confidence in that recommendation gone UP, DOWN, or
>   shifted sideways onto different axes?
> - What's the single piece of evidence that most influenced your
>   current position?
> - What would you need to know — evidence you don't currently have
>   — to change your mind?

**Evidence expected:** a reasoned position that synthesizes across
sections 1-4. No new experimental findings needed; this is a synthesis
question.

**YOUR RESPONSE:**




---
## Wrap-up
---

In [ ]:
# GUIDED: Save homework results for your records
homework_results = {
    "slice_table": slice_df.to_dict(orient="records"),
    "temperature_scaling": results_temp,
    "qualitative_tags": {
        "encoder_right_decoder_wrong": dict(tag_counts_enc),
        "decoder_right_encoder_wrong": dict(tag_counts_dec),
    },
    "per_class_worst": {
        "encoder": [{"class_id": int(c), "name": LABEL_LIST[c],
                     "f1": float(enc_pcf1[c])} for c in enc_worst],
        "decoder": [{"class_id": int(c), "name": LABEL_LIST[c],
                     "f1": float(dec_pcf1[c])} for c in dec_worst],
    },
    "bootstrap": {
        "encoder_mean": float(boot_enc.mean()),
        "encoder_ci95": [enc_lo, enc_hi],
        "decoder_mean": float(boot_dec.mean()),
        "decoder_ci95": [dec_lo, dec_hi],
        "diff_mean":    float(boot_diff.mean()),
        "diff_ci95":    [diff_lo, diff_hi],
        "diff_excludes_zero": bool(diff_excludes_zero),
    },
}

with open("week4_homework_results.json", "w") as f:
    json.dump(homework_results, f, indent=2)
print("Saved week4_homework_results.json")

### Submit

1. Run the notebook end-to-end with your answers filled in.
2. Export to HTML: **File → Download as → HTML (.html)** in Jupyter,
   or on Kaggle: the notebook already renders as HTML; click the
   "..." menu → "Save Version" to keep a static copy.
3. Upload the HTML to Moodle.

**Due:** Wednesday morning before class.